<a href="https://colab.research.google.com/github/ablackwellb/andean-heritage-network/blob/main/MET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## I. MET Openaccess Data Retrieval

In [ ]:
# install & imports
!pip install pyvis
from pyvis.network import Network
import json
import networkx as nx
import pandas as pd
import re
import seaborn as sns

# data retrieval
url = "https://media.githubusercontent.com/media/metmuseum/openaccess/master/MetObjects.csv"
df = pd.read_csv(url, low_memory=False)

print("Total rows:", len(df))
print("\nColumn names:")
for c in df.columns:
    print(" ", c)

Total rows: 484956

Column names:
  Object Number
  Is Highlight
  Is Timeline Work
  Is Public Domain
  Object ID
  Gallery Number
  Department
  AccessionYear
  Object Name
  Title
  Culture
  Period
  Dynasty
  Reign
  Portfolio
  Constituent ID
  Artist Role
  Artist Prefix
  Artist Display Name
  Artist Display Bio
  Artist Suffix
  Artist Alpha Sort
  Artist Nationality
  Artist Begin Date
  Artist End Date
  Artist Gender
  Artist ULAN URL
  Artist Wikidata URL
  Object Date
  Object Begin Date
  Object End Date
  Medium
  Dimensions
  Credit Line
  Geography Type
  City
  State
  County
  Country
  Region
  Subregion
  Locale
  Locus
  Excavation
  River
  Classification
  Rights and Reproduction
  Link Resource
  Object Wikidata URL
  Metadata Date
  Repository
  Tags
  Tags AAT URL
  Tags Wikidata URL


# II. Department Scan for Andean Cultures

In [ ]:
aoa = df[df["Department"] == "Arts of Africa, Oceania, and the Americas"]
print("Objects in that department:", len(aoa))

andean_terms = ["Inca", "Inka", "Wari", "Huari", "Chimú", "Chimu", "Chancay",
                "Nazca", "Nasca", "Paracas", "Moche", "Tiwanaku", "Tiahuanaco",
                "Chavín", "Chavin", "Lambayeque", "Sicán", "Sican"]

# case-insensitive match on Culture, treating blanks safely
mask = aoa["Culture"].fillna("").str.contains("|".join(andean_terms), case=False)
andean = aoa[mask]
print("Andean objects matched:", len(andean))

# how many of each culture?
print("\nTop cultures:")
print(andean["Culture"].value_counts().head(20))

# and how many true knot-records?
quipu = df[df["Object Name"].fillna("").str.contains("quipu|khipu", case=False) |
           df["Classification"].fillna("").str.contains("quipu|khipu", case=False)]
print("\nQuipu / khipu objects:", len(quipu))

Objects in that department: 12367
Andean objects matched: 2108

Top cultures:
Culture
Moche                 666
Paracas               396
Chimú                 217
Nasca                 184
Inca                  163
Wari                  125
Chimú or Chancay       83
Chancay                57
Chimú (?)              55
Lambayeque (Sicán)     48
Tiwanaku               24
Inca (?)               16
Moche (?)              13
Nasca (?)               8
Moche-Wari              6
Chimú or Inca           5
Chavin (?)              4
Chavin                  4
Lambayeque (?)          3
Tiwanaku (?)            3
Name: count, dtype: int64

Quipu / khipu objects: 0


# II. Survived Objects

>Object Construction

In [ ]:
print("Top object types:")
print(andean["Object Name"].fillna("Unknown").value_counts().head(15))

print("\nTop materials (Medium):")
print(andean["Medium"].fillna("Unknown").value_counts().head(15))

print("\nHow complete are our key fields? (non-blank counts of 2,108)")
for col in ["Culture", "Medium", "Period", "Region", "Classification", "Object Name"]:
    filled = andean[col].notna().sum()
    print(f"  {col}: {filled}")

Top object types:
Object Name
Ornament            389
Bottle              387
Bowl                221
Knife                68
Textile fragment     48
Beaker               46
Hat                  44
Pin                  44
Figure               42
Jar                  41
Panel                41
Vessel               38
Tunic                33
Border Fragment      33
Bag                  32
Name: count, dtype: int64

Top materials (Medium):
Medium
Ceramic, pigment           261
Ceramic                    206
Camelid hair               152
Camelid hair, cotton       120
Gilded copper              103
Gold                       102
Ceramic, slip              100
Silver                      98
Copper                      94
Ceramic, slip, pigment      80
Copper (cast)               65
Cotton, camelid hair        61
Silvered copper             48
Gilded copper, shell        35
Silver (hammered), gilt     34
Name: count, dtype: int64

How complete are our key fields? (non-blank counts of 2,108)

# II. Survived Objects
> **Quipu/Khipu:** *Does the Met hold ANY quipu, anywhere in the full collection??*

In [ ]:
q = df[df["Object Name"].fillna("").str.contains("quipu|khipu", case=False) |
       df["Title"].fillna("").str.contains("quipu|khipu", case=False) |
       df["Classification"].fillna("").str.contains("quipu|khipu", case=False)]
print("Quipu/khipu anywhere in the Met:", len(q))

Quipu/khipu anywhere in the Met: 0


# III. Methodology Verification

> **Material-first Textile Sweep:** Isolating every textile in the Andean subset to manually audit titles and ensure no quipu is misclassified.

In [ ]:
textile_mask = andean["Classification"].fillna("").str.contains("textile", case=False)
andean_textiles = andean[textile_mask]

print(f"Total Andean Textiles found: {len(andean_textiles)}")
print("Reviewing object titles to guarantee zero misclassified cord records:\n")

columns_to_inspect = ['Object ID', 'Title', 'Culture', 'Medium']
if not andean_textiles.empty:
    pd.set_option('display.max_rows', None)
    print(andean_textiles[columns_to_inspect].to_string(index=False))
    pd.reset_option('display.max_rows')
else:
    print("No objects found under the 'Textile' classification.")

Total Andean Textiles found: 430
Reviewing object titles to guarantee zero misclassified cord records:

 Object ID                                     Title            Culture                                                              Medium
    307448                                     Tunic              Chimú                                                Camelid hair, cotton
    307449                                       Bag               Inca                                                              Cotton
    307450                  Textile Fragment, Figure Lambayeque (Sicán)                                                Cotton, camelid hair
    307452                  Textile Fragment, Figure Lambayeque (Sicán)                                                Camelid hair, cotton
    307454                             Band Fragment          Chimú (?)                                                        Camelid hair
    307471                           Border Fragment    

# III. Methodology Verification

> **Audit:** Do the Textiles contain Khipu/Quipu?

In [ ]:
# filtering textiles to cite materials associated with khipu/quipu
def clean_material(m):
    m = m.strip().lower()
    m = re.sub(r"\s*\(.*\)", "", m)
    m = m.replace("post-fired", "post-fire")
    return m.strip()

# define materials commonly used for quipu/khipu
quipu_materials = ['cotton', 'camelid hair', 'camelid fiber']

# are cleaned material in a medium string which matches quipu_materials?
def contains_quipu_material(medium_str):
    if pd.isna(medium_str):
        return False
    cleaned_mats = [clean_material(m) for m in medium_str.split(',')]
    return any(mat in quipu_materials for mat in cleaned_mats)

# apply filter to andean_textiles
quipu_candidate_textiles = andean_textiles[andean_textiles['Medium'].apply(contains_quipu_material)]

print(f"Total Textile objects made from quipu-related materials found: {len(quipu_candidate_textiles)}")
print("Reviewing object titles of these candidates for potential misclassified quipu/khipu:\n")

columns_to_inspect = ['Object ID', 'Title', 'Culture', 'Medium']
if not quipu_candidate_textiles.empty:
    pd.set_option('display.max_rows', None) #
    print(quipu_candidate_textiles[columns_to_inspect].to_string(index=False))
    pd.reset_option('display.max_rows')
else:
    print("No textile objects found with quipu-related materials.")

Total Textile objects made from quipu-related materials found: 422
Reviewing object titles of these candidates for potential misclassified quipu/khipu:

 Object ID                                     Title            Culture                                                              Medium
    307448                                     Tunic              Chimú                                                Camelid hair, cotton
    307449                                       Bag               Inca                                                              Cotton
    307450                  Textile Fragment, Figure Lambayeque (Sicán)                                                Cotton, camelid hair
    307452                  Textile Fragment, Figure Lambayeque (Sicán)                                                Camelid hair, cotton
    307454                             Band Fragment          Chimú (?)                                                        Camelid hair
    307

# III. Methodology Verification

> **Global Archaic Synonym Scan:** Hunts all 484,956 records for historical
Eurocentric, or literal terms used for Quipus/Khipus before Modern Standardization.

In [ ]:
archaic_synonyms = [
    'knot-record', 'knot record', 'knotted cord', 'knotted string',
    'quipocamayoc', 'khipukamayuq', 'accounting device', 'peruvian thread',
    'talk knots', 'talking knot', 'assembled cords'
]

# create combined, lower-case text field across dataset
global_search_space = (
    df['Title'].fillna('') + " " +
    df['Object Name'].fillna('') + " " +
    df['Medium'].fillna('')
).str.lower()

# build a combined regex pattern to catch any of the synonyms
synonym_pattern = '|'.join(archaic_synonyms)
synonym_matches = df[global_search_space.str.contains(synonym_pattern, na=False)]

print(f"Global sweep completed across full collection.")
print(f"Total hits for archaic quipu terminology: {len(synonym_matches)}")

if not synonym_matches.empty:
    print("\nReviewing matches:")
    print(synonym_matches[['Object ID', 'Title', 'Object Name', 'Culture', 'Classification']].to_string(index=False))
else:
    print("\nSUCCESS: Absolute zero footprint. No hidden records found using legacy naming conventions.")


Global sweep completed across full collection.
Total hits for archaic quipu terminology: 0

SUCCESS: Absolute zero footprint. No hidden records found using legacy naming conventions.


# IV. Materials

*Which materials bridge cultures that are otherwise separate?*

In [ ]:
# multi-material Medium cells split into individual materials
materials = (
    andean["Medium"]
    .fillna("Unknown")
    .str.split(r",\s*")        # break "Camelid hair, cotton" into two
    .explode()                 # one row per object–material pair
    .str.strip()
    .str.lower()               # normalize case so "Gold" equals "gold"
)

print("Number of distinct materials after splitting:", materials.nunique())
print("\nTop 25 individual materials:")
print(materials.value_counts().head(25))

Number of distinct materials after splitting: 136

Top 25 individual materials:
Medium
ceramic               701
camelid hair          366
pigment               356
cotton                269
slip                  184
gilded copper         171
copper                153
gold                  147
silver                146
shell                 109
gilt                   89
silvered copper        88
silver (hammered)      76
copper (cast)          76
stone                  61
wood                   48
turquoise              34
copper (hammered)      30
feathers               28
post-fired paint       23
paint                  17
feathers on cotton     10
shell beads            10
post-fire paint        10
camelid fiber          10
Name: count, dtype: int64


# V. Imbalanced Material Clean-up

**Materials Found Contain Near-Duplicates**

*Examples:*

- Copper Variants: Copper(hammered) & Copper(cast)
- SilverCopper/GildedCopper
- Pigment/Paint

**Steps Taken:**

> Fix the Typos (i.e., "post-fire" vs. "post-fired"

> Keep the Variants

> Keep Material Types

> Combine Methods into Materials (i.e. "copper(cast)" into "copper")

In [ ]:
mats_clean = (
    andean["Medium"]
    .fillna("unknown")
    .str.split(r",\s*")
    .explode()
    .map(clean_material)
)
mats_clean = mats_clean[mats_clean != ""]

print("Distinct materials BEFORE cleaning: 136")
print("Distinct materials AFTER cleaning:", mats_clean.nunique())
print("\nTop 20 now:")
print(mats_clean.value_counts().head(20))

Distinct materials BEFORE cleaning: 136
Distinct materials AFTER cleaning: 109

Top 20 now:
Medium
ceramic               701
camelid hair          366
pigment               357
cotton                269
copper                260
silver                234
slip                  184
gilded copper         175
gold                  151
shell                 109
gilt                   94
silvered copper        89
stone                  61
wood                   54
turquoise              36
post-fire paint        33
feathers               28
paint                  17
camelid fiber          10
feathers on cotton     10
Name: count, dtype: int64


# VI. Network Construction

In [ ]:
from itertools import combinations

# compress bipartite graph into attribute-only co-occurrence map
H = nx.Graph()
for obj in [n for n in G if str(n).startswith("obj_")]:
    attrs = list(G.neighbors(obj))
    for a, b in combinations(attrs, 2):
        if H.has_edge(a, b):
            H[a][b]["weight"] += 1
        else:
            H.add_edge(a, b, weight=1)

H_clean = nx.Graph()
for u, v, d in H.edges(data=True):
    if d["weight"] >= 3:
        H_clean.add_edge(u, v, weight=d["weight"])

print("Projected graph:", H_clean.number_of_nodes(), "nodes |",
      H_clean.number_of_edges(), "edges")

Projected graph: 52 nodes | 171 edges


# VII. Visualization Construction

In [ ]:
# create the pyvis network from the networkx graph G
net = Network("800px", "1000px", notebook=True, cdn_resources="in_line",
              bgcolor="#111111", font_color="white")

for node in H_clean.nodes():
    kind, label = node.split("::", 1)
    score = attr_bt.get(node, 0)
    color = "#C44E52" if kind == "culture" else "#4C72B0"
    net.add_node(node, label=label, size=max(10, score * 300),
                 color=color, title=f"{label} — Betweenness Centrality: {score:.3f}")

for u, v, d in H_clean.edges(data=True):
    net.add_edge(u, v, width=min(d["weight"] * 0.1, 8),
                 color="rgba(200,200,200,0.15)")

net.barnes_hut(gravity=-8000, central_gravity=0.3, spring_length=150)
net.save_graph("Andean_Network_Betweenness_Centrality.html")
print("Saved — this one renders.")

Saved — this one renders.
